# 06 — Customer Intelligence
Turning customer-level features into business insights.
This notebook covers: behavior analysis, value analysis, segmentation, high-value customers, at-risk customers, geographic analysis, KPIs, and business insights.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:.2f}".format)

## 1. Load Customer Features

In [ ]:
from pathlib import Path

cf = pd.read_csv(Path("../data/processed/customer_features.csv"))

print("Shape:", cf.shape)
print("\nColumns:", cf.columns.tolist())
print("\nDtypes:\n", cf.dtypes)
print("\nMissing values:\n", cf.isna().sum())

In [ ]:
# verify customer_unique_id is truly unique (one row per customer)
assert cf["customer_unique_id"].nunique() == len(cf), "Duplicate customers found!"
print("customer_unique_id is unique. Total customers:", len(cf))

## 2. Customer Behavior Analysis

In [ ]:
# order count distribution
print(cf["order_count"].value_counts().head(10))
print("\nSummary:")
print(cf["order_count"].describe())

In [ ]:
# one-time vs repeat customers
cf["is_repeat"] = (cf["order_count"] > 1).astype(int)

repeat_counts = cf["is_repeat"].value_counts()
repeat_pct = cf["is_repeat"].value_counts(normalize=True) * 100

print("One-time customers: ", repeat_counts[0], f"({repeat_pct[0]:.1f}%)")
print("Repeat customers:   ", repeat_counts[1], f"({repeat_pct[1]:.1f}%)")

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["One-time", "Repeat"], repeat_counts.values, color=["#4C72B0", "#DD8452"])
ax.set_title("One-time vs Repeat Customers")
ax.set_ylabel("Customer Count")
plt.tight_layout()
plt.show()

In [ ]:
# spending distribution — cap at 99th percentile to avoid extreme outliers distorting the plot
spend_cap = cf["total_spend"].quantile(0.99)

fig, ax = plt.subplots(figsize=(8, 4))
cf[cf["total_spend"] <= spend_cap]["total_spend"].hist(bins=50, ax=ax, color="#4C72B0", edgecolor="white")
ax.set_title("Total Spend Distribution (capped at 99th percentile)")
ax.set_xlabel("Total Spend (BRL)")
ax.set_ylabel("Customer Count")
plt.tight_layout()
plt.show()

print(cf["total_spend"].describe())

In [ ]:
# recency distribution
fig, ax = plt.subplots(figsize=(8, 4))
cf["days_since_last_order"].hist(bins=50, ax=ax, color="#55A868", edgecolor="white")
ax.set_title("Days Since Last Order (Recency)")
ax.set_xlabel("Days")
ax.set_ylabel("Customer Count")
plt.tight_layout()
plt.show()

print(cf["days_since_last_order"].describe())

In [ ]:
# purchase frequency (if available)
if "purchase_frequency" in cf.columns:
    print(cf["purchase_frequency"].describe())
else:
    print("purchase_frequency column not found")

## 3. Customer Value Analysis
We use total_spend to segment customers into Low / Medium / High value.
Thresholds are based on the 33rd and 66th percentiles — data-driven, not arbitrary.

In [ ]:
low_thresh = cf["total_spend"].quantile(0.33)
high_thresh = cf["total_spend"].quantile(0.66)

print(f"Low threshold (33rd pct):  {low_thresh:.2f} BRL")
print(f"High threshold (66th pct): {high_thresh:.2f} BRL")

def value_segment(spend):
    if spend <= low_thresh:
        return "Low"
    elif spend <= high_thresh:
        return "Medium"
    else:
        return "High"

cf["value_segment"] = cf["total_spend"].apply(value_segment)
print("\nValue segment counts:")
print(cf["value_segment"].value_counts())

In [ ]:
seg_stats = cf.groupby("value_segment")[["total_spend", "order_count", "average_order_value"]].mean()
print(seg_stats)

fig, ax = plt.subplots(figsize=(6, 4))
order = ["Low", "Medium", "High"]
counts = cf["value_segment"].value_counts().reindex(order)
ax.bar(order, counts.values, color=["#4C72B0", "#DD8452", "#55A868"])
ax.set_title("Customer Value Segments")
ax.set_ylabel("Customer Count")
plt.tight_layout()
plt.show()

## 4. Customer Segmentation (RFM-style)
We segment customers using Recency, Frequency, and Monetary value.
Each dimension is scored 1-3 (low to high) using tertile splits.
Combined RFM score gives an interpretable segment label.
We use rule-based segmentation here — clustering is not needed at this stage because the business dimensions are already well-defined.

In [ ]:
# Recency: lower days_since_last_order = more recent = better = score 3
cf["R_score"] = pd.qcut(cf["days_since_last_order"], q=3, labels=[3, 2, 1]).astype(int)

# Frequency: higher order_count = better = score 3
# order_count has low variance (most customers = 1), use rank-based to handle ties
cf["F_score"] = pd.qcut(cf["order_count"].rank(method="first"), q=3, labels=[1, 2, 3]).astype(int)

# Monetary: higher total_spend = better = score 3
cf["M_score"] = pd.qcut(cf["total_spend"].rank(method="first"), q=3, labels=[1, 2, 3]).astype(int)

cf["RFM_score"] = cf["R_score"] + cf["F_score"] + cf["M_score"]

print(cf[["R_score", "F_score", "M_score", "RFM_score"]].describe())

In [ ]:
# map RFM total score to business segment labels
def rfm_label(score):
    if score >= 8:
        return "Champions"
    elif score >= 6:
        return "Loyal"
    elif score >= 4:
        return "Potential"
    else:
        return "At Risk"

cf["rfm_segment"] = cf["RFM_score"].apply(rfm_label)

print(cf["rfm_segment"].value_counts())

fig, ax = plt.subplots(figsize=(6, 4))
seg_order = ["Champions", "Loyal", "Potential", "At Risk"]
counts = cf["rfm_segment"].value_counts().reindex(seg_order)
ax.bar(seg_order, counts.values, color=["#55A868", "#4C72B0", "#DD8452", "#C44E52"])
ax.set_title("RFM Customer Segments")
ax.set_ylabel("Customer Count")
plt.tight_layout()
plt.show()

In [ ]:
# profile each segment
cf.groupby("rfm_segment")[["total_spend", "order_count", "days_since_last_order", "average_order_value"]].mean().round(2)

## 5. High-Value Customer Analysis

In [ ]:
hv = cf[cf["value_segment"] == "High"].copy()
print("High-value customers:", len(hv))
print("\nProfile:")
print(hv[["total_spend", "order_count", "days_since_last_order", "average_order_value"]].describe().round(2))

In [ ]:
# what proportion of high-value customers are repeat buyers?
hv_repeat = hv["is_repeat"].mean() * 100
all_repeat = cf["is_repeat"].mean() * 100
print(f"Repeat rate - High-value: {hv_repeat:.1f}%")
print(f"Repeat rate - All customers: {all_repeat:.1f}%")

In [ ]:
# geographic distribution of high-value customers
hv_state = hv["customer_state"].value_counts().head(10)
print(hv_state)

fig, ax = plt.subplots(figsize=(8, 4))
hv_state.plot(kind="bar", ax=ax, color="#4C72B0")
ax.set_title("High-Value Customers by State (Top 10)")
ax.set_ylabel("Customer Count")
ax.set_xlabel("State")
plt.tight_layout()
plt.show()

## 6. At-Risk / Inactive Customer Analysis
These are rule-based observations using recency and purchase behavior.
This is NOT churn prediction. We have not defined a supervised ML target yet.
Customers inactive for 180+ days with only 1 order are flagged as potentially at-risk.

In [ ]:
# rule: inactive 180+ days AND only 1 order
# 180 days chosen because the dataset spans ~2 years; 6 months of inactivity is a meaningful signal
at_risk = cf[(cf["days_since_last_order"] >= 180) & (cf["order_count"] == 1)].copy()

print("At-risk / inactive customers:", len(at_risk))
print(f"Percentage of all customers: {len(at_risk)/len(cf)*100:.1f}%")
print("\nProfile:")
print(at_risk[["total_spend", "days_since_last_order", "average_order_value"]].describe().round(2))

In [ ]:
# value breakdown of at-risk customers
# important: are we losing high-value customers or mostly low-value?
print(at_risk["value_segment"].value_counts())
print("\nAverage spend of at-risk customers:", at_risk["total_spend"].mean().round(2))

## 7. Geographic Customer Analysis

In [ ]:
geo = cf.groupby("customer_state").agg(
    customer_count=("customer_unique_id", "count"),
    total_spend=("total_spend", "sum"),
    avg_spend=("total_spend", "mean"),
    repeat_rate=("is_repeat", "mean")
).sort_values("customer_count", ascending=False).round(2)

print(geo.head(10))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top10 = geo.head(10)

top10["customer_count"].plot(kind="bar", ax=axes[0], color="#4C72B0")
axes[0].set_title("Customer Count by State (Top 10)")
axes[0].set_ylabel("Customers")
axes[0].set_xlabel("State")

top10["avg_spend"].plot(kind="bar", ax=axes[1], color="#55A868")
axes[1].set_title("Avg Customer Spend by State (Top 10)")
axes[1].set_ylabel("Avg Spend (BRL)")
axes[1].set_xlabel("State")

plt.tight_layout()
plt.show()

## 8. Customer Intelligence KPIs

In [ ]:
total_customers = len(cf)
one_time_pct = (cf["order_count"] == 1).mean() * 100
repeat_pct = (cf["order_count"] > 1).mean() * 100
avg_spend = cf["total_spend"].mean()
median_spend = cf["total_spend"].median()
avg_orders = cf["order_count"].mean()
at_risk_count = len(at_risk)
at_risk_pct = len(at_risk) / total_customers * 100
hv_count = len(hv)
hv_pct = len(hv) / total_customers * 100

kpis = {
    "Total Customers": total_customers,
    "One-time Customer %": f"{one_time_pct:.1f}%",
    "Repeat Customer %": f"{repeat_pct:.1f}%",
    "Avg Customer Spend (BRL)": round(avg_spend, 2),
    "Median Customer Spend (BRL)": round(median_spend, 2),
    "Avg Orders per Customer": round(avg_orders, 3),
    "At-Risk Customers": f"{at_risk_count} ({at_risk_pct:.1f}%)",
    "High-Value Customers": f"{hv_count} ({hv_pct:.1f}%)",
}

for k, v in kpis.items():
    print(f"{k}: {v}")

## 9. Business Insights
Summary of the strongest customer-level findings.

In [ ]:
print("=== CUSTOMER INTELLIGENCE SUMMARY ===\n")

print("WHAT IS HAPPENING?")
print(f"- {one_time_pct:.1f}% of customers placed only one order. Repeat purchase rate is very low.")
print(f"- Average customer spend is {avg_spend:.2f} BRL, but median is {median_spend:.2f} BRL — spend is right-skewed.")
print(f"- {at_risk_pct:.1f}% of customers are inactive (180+ days, single order) — potential retention candidates.")
print(f"- High-value customers ({hv_pct:.1f}% of base) drive a disproportionate share of revenue.")

print("\nWHY MIGHT IT BE HAPPENING? (observed patterns, not proven causes)")
print("- The low repeat rate may reflect the nature of the product mix (one-off purchases).")
print("- High recency values suggest many customers have not returned since their first purchase.")
print("- SP state dominates both customer count and spend — geographic concentration is high.")

print("\nWHAT SHOULD THE BUSINESS CONSIDER?")
print("- Prioritize retention campaigns for at-risk high-value customers.")
print("- Investigate what drives the small repeat-buyer segment — replicate those conditions.")
print("- Consider region-specific strategies given the geographic concentration in SP.")

## 10. Prepare for Later ML
Features that could be useful for future churn/retention modeling.
No model is trained here. No churn target is defined here.

In [ ]:
ml_candidates = [
    "order_count",           # frequency signal
    "total_spend",           # monetary signal
    "average_order_value",   # spend per order
    "days_since_last_order", # recency signal
    "purchase_frequency",    # orders per period
    "avg_days_between_orders", # engagement cadence
    "unique_categories",     # breadth of purchasing
    "avg_review_score",      # satisfaction signal
    "late_order_rate",       # delivery experience
    "customer_state",        # geographic feature (needs encoding)
]

available = [c for c in ml_candidates if c in cf.columns]
missing = [c for c in ml_candidates if c not in cf.columns]

print("Available for ML:", available)
print("Not yet in dataset:", missing)
print("\nNote: churn target variable will be defined in the next notebook.")